# Part 4: MLP 模型训练及评估

请先运行 `01_dependencies_and_data.ipynb`。

In [1]:
"""Part 4: MLP. Run 01_dependencies_and_data.ipynb first."""
import os, random, sys
import dill
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

root_dir    = "C:\ML4GM"
outputs_dir = os.path.join(root_dir, "models")

def load_pkl(filepath):
    with open(filepath, "rb") as fr: return dill.load(fr)
def save_pkl(filepath, data):
    with open(filepath, "wb") as fw: dill.dump(data, fw)
    print(f"[{filepath}] data saved.")

_data = load_pkl(os.path.join(outputs_dir, "preprocessed_data.pkl"))
# 强制转为普通 ndarray（防止 memmap 在 Windows loky 子进程中无法访问）
X_all       = np.asarray(_data["X_all"],     dtype=np.float64)
y_all       = np.asarray(_data["y_all"],     dtype=np.float64)
rgiid_all   = np.asarray(_data["rgiid_all"])
year_all    = np.asarray(_data["year_all"])
feature_columns = _data["feature_columns"]

print(f"Loaded: X_all={X_all.shape}, y_all={y_all.shape}, years={sorted(np.unique(year_all))}")
print(f"X_all type: {type(X_all)}, dtype: {X_all.dtype}")


Loaded: X_all=(162020, 46), y_all=(162020,), years=[np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
X_all type: <class 'numpy.ndarray'>, dtype: float64


### MLP 模型训练及评估

In [2]:
def setup_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    torch.use_deterministic_algorithms(True)

setup_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y).unsqueeze(1)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


class MLPModel(nn.Module):
    """多层感知机：Input → [256→PReLU→Drop → 128→PReLU→Drop → 64→PReLU→Drop] → 1"""
    def __init__(self, input_size, hidden_sizes=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers, in_s = [], input_size
        for h in hidden_sizes:
            layers += [nn.Linear(in_s, h), nn.PReLU(), nn.Dropout(dropout)]
            in_s = h
        layers.append(nn.Linear(in_s, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)


def predict_batched(model, X_np, bs=4096):
    """对大数组分批推断，返回 np.ndarray (N,)"""
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X_np), bs):
            X_b = torch.FloatTensor(X_np[i:i+bs]).to(device)
            preds.append(model(X_b).cpu().numpy().flatten())
    return np.concatenate(preds)


Device: cpu


In [ ]:
import itertools
from sklearn.model_selection import KFold

# ---- MLP 超参数网格搜索（Grid Search）----
# 网格设计依据：
#   - 论文结构为 512→256→128，dropout=0.3 → 以此为中心扩展上下两档
#   - 162K 大样本支持 512 宽层，不易欠拟合
#   - lr 覆盖 1e-4~1e-3；weight_decay 两档正则化强度
TUNING_EPOCHS = 50   # 调参阶段快速估算轮数
TUNING_FOLDS  = 3    # 使用前 3 折（节省时间）

param_grid_mlp = {
    "hidden_sizes":  [(512, 256, 128), (256, 128, 64), (512, 256, 64)],
    "dropout":       [0.2, 0.3, 0.4],
    "learning_rate": [1e-4, 5e-4, 1e-3],
    "weight_decay":  [1e-5, 1e-4],
}
# 3×3×3×2 = 54 组合

kf        = KFold(n_splits=5, shuffle=True, random_state=42)
cv_splits = list(kf.split(X_all, y_all))[:TUNING_FOLDS]

keys   = list(param_grid_mlp.keys())
combos = list(itertools.product(*param_grid_mlp.values()))
print(f"[MLP 调参] 网格大小: {len(combos)} 组合 × {TUNING_FOLDS} 折")

best_score, best_params_mlp = float("-inf"), {}
for ci, combo in enumerate(combos):
    params = dict(zip(keys, combo))
    fold_scores = []
    for fi, (tr_idx, va_idx) in enumerate(cv_splits):
        X_tr, y_tr_ = X_all[tr_idx], y_all[tr_idx]
        X_va, y_va  = X_all[va_idx],  y_all[va_idx]
        sc = StandardScaler()
        X_tr_s, X_va_s = sc.fit_transform(X_tr), sc.transform(X_va)
        setup_seed(42)
        m    = MLPModel(X_tr_s.shape[1], params["hidden_sizes"], params["dropout"]).to(device)
        opt_ = torch.optim.Adam(m.parameters(),
                                lr=params["learning_rate"], weight_decay=params["weight_decay"])
        crit_= nn.MSELoss()
        ldr  = DataLoader(TabularDataset(X_tr_s, y_tr_),
                          batch_size=512, shuffle=True, num_workers=0)
        for _ in range(TUNING_EPOCHS):
            m.train()
            for X_b, y_b in ldr:
                out = m(X_b.to(device))
                loss = crit_(out, y_b.to(device))
                opt_.zero_grad(); loss.backward(); opt_.step()
        fold_scores.append(r2_score(y_va, predict_batched(m, X_va_s)))
    mean_r2 = float(np.mean(fold_scores))
    print(f"  [{ci+1:3d}/{len(combos)}] hidden={params['hidden_sizes']} "
          f"drop={params['dropout']:.1f} lr={params['learning_rate']:.0e} "
          f"wd={params['weight_decay']:.0e} → R²={mean_r2:.4f}")
    if mean_r2 > best_score:
        best_score      = mean_r2
        best_params_mlp = params

hidden_sizes     = best_params_mlp["hidden_sizes"]
dropout          = best_params_mlp["dropout"]
learning_rate    = best_params_mlp["learning_rate"]
weight_decay_val = best_params_mlp["weight_decay"]
batch_size       = 512
epochs           = 500
patience         = 30
print(f"\n[MLP 调参] Best R²={best_score:.4f}")
print(f"  hidden_sizes={hidden_sizes}, dropout={dropout:.2f}, "
      f"lr={learning_rate:.2e}, weight_decay={weight_decay_val:.2e}")


In [ ]:
# 数据准备：在全量数据上 90/10 分割，10% 用于 early stopping
scaler_full = StandardScaler()
X_all_std   = scaler_full.fit_transform(X_all)

n_all = len(X_all_std)
rng   = np.random.RandomState(42)
va_idx = rng.choice(n_all, size=int(n_all * 0.1), replace=False)
tr_idx = np.setdiff1d(np.arange(n_all), va_idx)

tr_loader = DataLoader(TabularDataset(X_all_std[tr_idx], y_all[tr_idx]),
                       batch_size=batch_size, shuffle=True,  num_workers=0)
va_loader = DataLoader(TabularDataset(X_all_std[va_idx], y_all[va_idx]),
                       batch_size=batch_size, shuffle=False, num_workers=0)

_val_label = "Val(10%)"
print(f"Train: {len(tr_idx)}, Val: {len(va_idx)}")


In [ ]:
setup_seed(42)
mlp_model = MLPModel(X_all.shape[-1], hidden_sizes, dropout).to(device)
optimizer  = torch.optim.Adam(mlp_model.parameters(), lr=learning_rate, weight_decay=weight_decay_val)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
criterion  = nn.MSELoss()

best_val_r2, patience_counter = float("-inf"), 0
train_r2_records, train_loss_records = [], []
val_r2_records,   val_loss_records   = [], []

for epoch in range(1, epochs + 1):
    mlp_model.train()
    tr_real, tr_pred, tr_losses = [], [], []
    for X_b, y_b in tr_loader:
        out  = mlp_model(X_b.to(device))
        loss = criterion(out, y_b.to(device))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tr_real.extend(y_b.numpy().flatten()); tr_pred.extend(out.detach().cpu().numpy().flatten())
        tr_losses.append(loss.item())
    tr_r2   = round(r2_score(tr_real, tr_pred), 4)
    tr_loss = round(float(np.mean(tr_losses)), 6)

    mlp_model.eval()
    va_real, va_pred, va_losses = [], [], []
    with torch.no_grad():
        for X_b, y_b in va_loader:
            out  = mlp_model(X_b.to(device))
            loss = criterion(out, y_b.to(device))
            va_real.extend(y_b.numpy().flatten()); va_pred.extend(out.cpu().numpy().flatten())
            va_losses.append(loss.item())
    va_r2   = round(r2_score(va_real, va_pred), 4)
    va_loss = round(float(np.mean(va_losses)), 6)
    scheduler.step(va_loss)

    print(f"Epoch {epoch}/{epochs} | Train R2={tr_r2:.4f} Loss={tr_loss:.6f} | "
          f"{_val_label} R2={va_r2:.4f} Loss={va_loss:.6f}")
    train_r2_records.append(tr_r2);   train_loss_records.append(tr_loss)
    val_r2_records.append(va_r2);     val_loss_records.append(va_loss)

    if va_r2 > best_val_r2:
        best_val_r2 = va_r2; patience_counter = 0
        torch.save(mlp_model.state_dict(), os.path.join(outputs_dir, "mlp_best.pt"))
    else:
        patience_counter += 1
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch}"); break

print(f"Done! Best {_val_label} R2: {best_val_r2:.4f}")

In [ ]:
mlp_model.load_state_dict(torch.load(os.path.join(outputs_dir, "mlp_best.pt"), map_location=device))
mlp_model.eval()
save_pkl(os.path.join(outputs_dir, "mlp_model.pkl"), mlp_model)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)
axes[0].plot(train_r2_records, label="Train R2")
axes[0].plot(val_r2_records,   label=f"{_val_label} R2")
axes[0].set_title("R2 during training (MLP)"); axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("R2"); axes[0].legend()
axes[1].plot(train_loss_records, label="Train Loss")
axes[1].plot(val_loss_records,   label=f"{_val_label} Loss")
axes[1].set_title("Loss during training (MLP)"); axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss"); axes[1].legend()
plt.tight_layout(); plt.show(); plt.close()


### LOYO 留一年交叉验证（Leave-One-Year-Out）

每折留出 1 个年份作为测试集，其余所有年份（所有冰川）作为训练集，共 20 折。  
每折从训练集中随机抽取 10% 作为 Early Stopping 验证集（不影响最终评估）。  
参考：Bolibar et al. (2020), *The Cryosphere*, 14, 565–584.

In [ ]:
unique_years = sorted(np.unique(year_all))
loyo_rows_ann = []
all_y_true, all_y_pred = [], []   # OOF 收集

for fold_i, fold_year in enumerate(unique_years, 1):
    print(f"\n[LOYO-ANN] ===== Fold {fold_i:2d}/20 | Held-out year: {fold_year} =====")
    tr_mask = year_all != fold_year
    te_mask = year_all == fold_year

    X_tr, y_tr = X_all[tr_mask], y_all[tr_mask]
    X_te, y_te = X_all[te_mask], y_all[te_mask]

    sc = StandardScaler()
    X_tr_std = sc.fit_transform(X_tr)
    X_te_std = sc.transform(X_te)

    # 从训练集中随机留 10% 用于 early stopping（不参与测试评估）
    n_tr = len(X_tr_std)
    rng  = np.random.RandomState(42)
    va_idx = rng.choice(n_tr, size=int(n_tr * 0.1), replace=False)
    tr_idx = np.setdiff1d(np.arange(n_tr), va_idx)

    tr_ldr = DataLoader(TabularDataset(X_tr_std[tr_idx], y_tr[tr_idx]),
                        batch_size=512, shuffle=True,  num_workers=0)
    va_ldr = DataLoader(TabularDataset(X_tr_std[va_idx], y_tr[va_idx]),
                        batch_size=512, shuffle=False, num_workers=0)

    setup_seed(42)
    fold_model = MLPModel(X_tr_std.shape[1], hidden_sizes, dropout).to(device)
    opt  = torch.optim.Adam(fold_model.parameters(), lr=learning_rate, weight_decay=weight_decay_val)
    sch  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=10, factor=0.5)
    crit = nn.MSELoss()

    best_val_r2_fold, patience_counter_fold, best_state_fold = float("-inf"), 0, None

    for epoch in range(1, 501):
        fold_model.train()
        for X_b, y_b in tr_ldr:
            out  = fold_model(X_b.to(device))
            loss = crit(out, y_b.to(device))
            opt.zero_grad(); loss.backward(); opt.step()

        fold_model.eval()
        va_real_f, va_pred_f, va_losses_f = [], [], []
        with torch.no_grad():
            for X_b, y_b in va_ldr:
                out = fold_model(X_b.to(device))
                va_real_f.extend(y_b.numpy().flatten())
                va_pred_f.extend(out.cpu().numpy().flatten())
                va_losses_f.append(crit(out, y_b.to(device)).item())
        va_r2_f   = r2_score(va_real_f, va_pred_f)
        sch.step(float(np.mean(va_losses_f)))

        if va_r2_f > best_val_r2_fold:
            best_val_r2_fold      = va_r2_f
            patience_counter_fold = 0
            best_state_fold       = {k: v.clone() for k, v in fold_model.state_dict().items()}
        else:
            patience_counter_fold += 1
        if patience_counter_fold >= 30:
            print(f"  Early stopping at epoch {epoch}, best val R²={best_val_r2_fold:.4f}")
            break

    fold_model.load_state_dict(best_state_fold)
    y_pred = predict_batched(fold_model, X_te_std)

    all_y_true.extend(y_te.tolist())
    all_y_pred.extend(y_pred.tolist())

    r2   = float(r2_score(y_te, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_te, y_pred)))
    mae  = float(mean_absolute_error(y_te, y_pred))

    loyo_rows_ann.append({
        "year": fold_year, "n_train": int(tr_mask.sum()),
        "n_test": int(te_mask.sum()), "R2": r2, "RMSE": rmse, "MAE": mae,
    })
    print(f"[LOYO-ANN] Year {fold_year} | n_test={te_mask.sum()} | "
          f"R2={r2:.4f} | RMSE={rmse:.4f} | MAE={mae:.4f}")

loyo_ann_df = pd.DataFrame(loyo_rows_ann)
loyo_ann_path = os.path.join(outputs_dir, "ann_loyo_results.csv")
loyo_ann_df.to_csv(loyo_ann_path, index=False)

print(f"\n[LOYO-ANN] ===== Summary ({len(unique_years)} folds) =====")
print(f"  R²   mean={loyo_ann_df['R2'].mean():.4f}  std={loyo_ann_df['R2'].std():.4f}  "
      f"min={loyo_ann_df['R2'].min():.4f}  max={loyo_ann_df['R2'].max():.4f}")
print(f"  RMSE mean={loyo_ann_df['RMSE'].mean():.4f}  std={loyo_ann_df['RMSE'].std():.4f}")
print(f"  MAE  mean={loyo_ann_df['MAE'].mean():.4f}  std={loyo_ann_df['MAE'].std():.4f}")
print(f"  Saved to: {loyo_ann_path}")

fig, axes = plt.subplots(1, 2, figsize=(16, 5), dpi=100)
axes[0].bar(loyo_ann_df["year"], loyo_ann_df["R2"], color="steelblue", alpha=0.8)
axes[0].axhline(loyo_ann_df["R2"].mean(), color="r", linestyle="--",
                label=f"Mean R²={loyo_ann_df['R2'].mean():.4f}")
axes[0].set_xlabel("Held-out Year", fontsize=13)
axes[0].set_ylabel("R²", fontsize=13)
axes[0].set_title("LOYO — Per-Year R² (MLP/ANN)", fontsize=15)
axes[0].legend()

axes[1].bar(loyo_ann_df["year"], loyo_ann_df["RMSE"], color="darkorange", alpha=0.8)
axes[1].axhline(loyo_ann_df["RMSE"].mean(), color="r", linestyle="--",
                label=f"Mean RMSE={loyo_ann_df['RMSE'].mean():.4f}")
axes[1].set_xlabel("Held-out Year", fontsize=13)
axes[1].set_ylabel("RMSE (m/yr)", fontsize=13)
axes[1].set_title("LOYO — Per-Year RMSE (MLP/ANN)", fontsize=15)
axes[1].legend()

plt.tight_layout()
plt.show()
plt.close()

# ---- OOF 散点图（Bolibar et al. 2020 Figure 9 风格） ----
all_y_true_np = np.array(all_y_true)
all_y_pred_np = np.array(all_y_pred)
r2_oof   = float(r2_score(all_y_true_np, all_y_pred_np))
rmse_oof = float(np.sqrt(mean_squared_error(all_y_true_np, all_y_pred_np)))
mae_oof  = float(mean_absolute_error(all_y_true_np, all_y_pred_np))

fig, ax = plt.subplots(figsize=(7, 7), dpi=100)
ax.scatter(all_y_true_np, all_y_pred_np, alpha=0.15, s=8, c="steelblue")
lims = [float(all_y_true_np.min()), float(all_y_true_np.max())]
ax.plot(lims, lims, "r--", linewidth=1.5)
ax.set_xlabel("Observed dhdt (m/yr)", fontsize=13)
ax.set_ylabel("Predicted dhdt (m/yr)", fontsize=13)
ax.set_title(f"LOYO Out-of-Fold Predictions (MLP/ANN)\n"
             f"R²={r2_oof:.4f}  RMSE={rmse_oof:.4f}  MAE={mae_oof:.4f}", fontsize=13)
plt.tight_layout()
plt.show()
plt.close()
print(f"[LOYO-ANN OOF] R²={r2_oof:.4f}  RMSE={rmse_oof:.4f}  MAE={mae_oof:.4f}  n={len(all_y_true_np)}")

### 方案一：5-fold 空间 GroupKFold CV

按冰川 ID 分组（GroupKFold），每折 80% 冰川（含其全部年份）作为训练集，20% 冰川作为测试集。  
**保证同一冰川的所有年份不会同时出现在训练集和测试集中，消除空间泄露。**  
每折独立拟合 StandardScaler，并从训练集再拨 10% 用于 early stopping（patience=20，最多 200 轮）。

In [ ]:
from sklearn.model_selection import KFold

CV_EPOCHS   = 200
CV_PATIENCE = 20

# ── 辅助函数：冰川/年份折编号 ──────────────────────────────────────────
def make_glacier_fold(rgiid_arr, n_folds=5):
    glacier_sorted = np.sort(np.unique(rgiid_arr))
    n_glaciers = len(glacier_sorted)
    gid_to_fold = {gid: int(i * n_folds // n_glaciers)
                   for i, gid in enumerate(glacier_sorted)}
    return np.array([gid_to_fold[g] for g in rgiid_arr], dtype=int)

def make_year_fold(year_arr, n_folds=5):
    year_sorted = np.sort(np.unique(year_arr))
    n_years = len(year_sorted)
    yr_to_fold = {yr: int(i * n_folds // n_years)
                  for i, yr in enumerate(year_sorted)}
    return np.array([yr_to_fold[y] for y in year_arr], dtype=int)

def train_fold_ann(X_tr_s, y_tr, X_te_s, y_te, fold_label):
    """训练一折 ANN，返回 (r2, rmse, mae)"""
    n_tr = len(X_tr_s)
    rng_ = np.random.RandomState(42)
    va_i = rng_.choice(n_tr, size=int(n_tr * 0.1), replace=False)
    tr_i = np.setdiff1d(np.arange(n_tr), va_i)

    tr_ldr = DataLoader(TabularDataset(X_tr_s[tr_i], y_tr[tr_i]),
                        batch_size=512, shuffle=True,  num_workers=0)
    setup_seed(42)
    m    = MLPModel(X_tr_s.shape[1], hidden_sizes, dropout).to(device)
    opt_ = torch.optim.Adam(m.parameters(), lr=learning_rate, weight_decay=weight_decay_val)
    sch_ = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_, patience=5, factor=0.5)
    crit_= nn.MSELoss()

    best_va, pat_, best_st = float("-inf"), 0, None
    for ep in range(1, CV_EPOCHS + 1):
        m.train()
        for X_b, y_b in tr_ldr:
            out  = m(X_b.to(device))
            loss = crit_(out, y_b.to(device))
            opt_.zero_grad(); loss.backward(); opt_.step()
        va_preds = predict_batched(m, X_tr_s[va_i])
        va_r2    = r2_score(y_tr[va_i], va_preds)
        sch_.step(-va_r2)
        if va_r2 > best_va:
            best_va = va_r2; pat_ = 0
            best_st = {k: v.cpu().clone() for k, v in m.state_dict().items()}
        else:
            pat_ += 1
        if pat_ >= CV_PATIENCE:
            print(f"  {fold_label}: early stop @ epoch {ep}, val R²={best_va:.4f}")
            break
    m.load_state_dict(best_st)
    y_pred = predict_batched(m, X_te_s)
    r2   = float(r2_score(y_te, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_te, y_pred)))
    mae  = float(mean_absolute_error(y_te, y_pred))
    return r2, rmse, mae

# ── 5-fold 随机 KFold CV ──────────────────────────────────────────────
kf_cv = KFold(n_splits=5, shuffle=True, random_state=42)
kfold_rows = []

for fold_i, (tr_idx, te_idx) in enumerate(kf_cv.split(X_all, y_all), 1):
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_all[tr_idx]).astype(np.float32)
    X_te_s = sc.transform(X_all[te_idx]).astype(np.float32)
    y_tr   = y_all[tr_idx].astype(np.float32)
    y_te   = y_all[te_idx].astype(np.float32)

    print(f"\n[ANN KFold CV] Fold {fold_i}/5 | n_train={len(tr_idx)} | n_test={len(te_idx)}")
    r2, rmse, mae = train_fold_ann(X_tr_s, y_tr, X_te_s, y_te, f"Fold{fold_i}")

    kfold_rows.append({
        "fold": fold_i, "n_train": len(tr_idx), "n_test": len(te_idx),
        "R2": r2, "RMSE": rmse, "MAE": mae,
    })
    print(f"[ANN KFold CV] Fold {fold_i}/5 | R2={r2:.4f} | RMSE={rmse:.4f} | MAE={mae:.4f}")

kfold_df = pd.DataFrame(kfold_rows)
kf_path = os.path.join(outputs_dir, "ann_kfold_cv_results.csv")
kfold_df.to_csv(kf_path, index=False)

print(f"\n[ANN KFold CV] ===== Summary =====")
print(f"  R²   mean={kfold_df['R2'].mean():.4f}  std={kfold_df['R2'].std():.4f}")
print(f"  RMSE mean={kfold_df['RMSE'].mean():.4f}  std={kfold_df['RMSE'].std():.4f}")
print(f"  MAE  mean={kfold_df['MAE'].mean():.4f}  std={kfold_df['MAE'].std():.4f}")
print(f"  Saved → {kf_path}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=100)
for ax, metric, color in zip(axes, ["R2", "RMSE", "MAE"],
                              ["steelblue", "darkorange", "seagreen"]):
    vals = kfold_df[metric]
    ax.bar(kfold_df["fold"], vals, color=color, alpha=0.8)
    ax.axhline(vals.mean(), color="r", linestyle="--", label=f"Mean={vals.mean():.4f}")
    ax.set_xlabel("Fold", fontsize=12); ax.set_ylabel(metric, fontsize=12)
    ax.set_title(f"ANN KFold CV — {metric}", fontsize=13); ax.legend()
plt.tight_layout(); plt.show(); plt.close()


### 方案二：5×5 Block CV（对角线）

冰川按字典序分 5 组，年份按时间顺序分 5 组，对角线块（冰川组 i ∩ 年份组 i）轮流作为测试集。  
训练集排除测试块所在的**整行**（同冰川组所有年份）和**整列**（同年份组所有冰川），约占 64%。  
**同时消除空间泄露和时间泄露**，评估模型对未见过的冰川×时期组合的时空双重泛化能力。

In [ ]:
# ── 5×5 Block CV（对角线） ────────────────────────────────────────────
glacier_fold = make_glacier_fold(rgiid_all, n_folds=5)
year_fold    = make_year_fold(year_all,    n_folds=5)

block_rows = []

for f in range(5):
    test_mask  = (glacier_fold == f) & (year_fold == f)
    train_mask = ~((glacier_fold == f) | (year_fold == f))

    assert not np.any(glacier_fold[train_mask] == f), \
        f"Block fold {f}: glacier leakage in train!"
    assert not np.any(year_fold[train_mask] == f), \
        f"Block fold {f}: year leakage in train!"

    if test_mask.sum() == 0:
        print(f"[ANN Block CV] Fold {f}: empty test set, skipping.")
        continue

    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_all[train_mask]).astype(np.float32)
    X_te_s = sc.transform(X_all[test_mask]).astype(np.float32)
    y_tr   = y_all[train_mask].astype(np.float32)
    y_te   = y_all[test_mask].astype(np.float32)

    print(f"\n[ANN Block CV] Fold {f}/4 | n_train={train_mask.sum()} | n_test={test_mask.sum()}")
    r2, rmse, mae = train_fold_ann(X_tr_s, y_tr, X_te_s, y_te, f"Block{f}")
    n_test_glaciers = len(np.unique(rgiid_all[test_mask]))

    block_rows.append({
        "fold": f, "n_train": int(train_mask.sum()), "n_test": int(test_mask.sum()),
        "n_test_glaciers": n_test_glaciers, "R2": r2, "RMSE": rmse, "MAE": mae,
    })
    print(f"[ANN Block CV] Fold {f}/4 | glaciers={n_test_glaciers} | "
          f"R2={r2:.4f} | RMSE={rmse:.4f} | MAE={mae:.4f}")

block_df = pd.DataFrame(block_rows)
blk_path = os.path.join(outputs_dir, "ann_block_cv_results.csv")
block_df.to_csv(blk_path, index=False)

print(f"\n[ANN Block CV] ===== Summary =====")
print(f"  R²   mean={block_df['R2'].mean():.4f}  std={block_df['R2'].std():.4f}")
print(f"  RMSE mean={block_df['RMSE'].mean():.4f}  std={block_df['RMSE'].std():.4f}")
print(f"  MAE  mean={block_df['MAE'].mean():.4f}  std={block_df['MAE'].std():.4f}")
print(f"  Saved → {blk_path}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=100)
for ax, metric, color in zip(axes, ["R2", "RMSE", "MAE"],
                              ["steelblue", "darkorange", "seagreen"]):
    vals = block_df[metric]
    ax.bar(block_df["fold"], vals, color=color, alpha=0.8)
    ax.axhline(vals.mean(), color="r", linestyle="--", label=f"Mean={vals.mean():.4f}")
    ax.set_xlabel("Fold", fontsize=12); ax.set_ylabel(metric, fontsize=12)
    ax.set_title(f"ANN Block CV — {metric}", fontsize=13); ax.legend()
plt.tight_layout(); plt.show(); plt.close()